In [ ]:
import pymaid
import navis

**Export neurons from CATMAID**

In [ ]:
def export_neurons(server, api_token, project_id, min_nodes=0, annotations=None, skids=None, reviewed_by=None):
    #Load instance and set project
    rm = pymaid.CatmaidInstance(server=server, api_token=api_token)
    rm.project_id = project_id

    #format annotations
    if annotations:
        annot_list = []
        for ann in annotations:
            annot_list.append('annotation:'+str(ann))
        annotations = annot_list

    skels = pymaid.find_neurons(min_size=min_nodes, annotations=annotations, skids=skids, reviewed_by=reviewed_by)

    return skels

In [ ]:
skels = export_neurons( server="http://bigkahuna:4554/", api_token='7deede658f4bc1f53ee6993fc28ec26fbbe5838f', project_id = '41', 
                       min_nodes=100, annotations = None, skids=None, reviewed_by=None)

**Import neurons into CATMAID**

In [ ]:
def import_neurons(skels, server, api_token, project_id, rad_annot=True, res=[1,1,1]):
    #Load instance and set project
    rm = pymaid.CatmaidInstance(server=server, api_token=api_token)
    rm.project_id = project_id

    #Import neurons
    for sk in skels:
        nodes = sk.nodes.copy()
        nodes = nodes[['node_id', 'parent_id', 'x', 'y', 'z', 'radius']]
        neu = navis.TreeNeuron(nodes)
        if res != [1,1,1]:
            x,y,z = res
            neu.nodes['x'] = neu.nodes['x']*x
            neu.nodes['y'] = neu.nodes['y']*y
            neu.nodes['z'] = neu.nodes['z']*z
        resp = pymaid.upload_neuron(neu)
        if rad_annot==True:
            rad = round(max(list(neu.nodes['radius'])))
            resp = pymaid.add_annotations(int(resp['skeleton_id']), "RADIUS:" + str(rad))

In [ ]:
skels = navis.read_swc("/ACdata/Users/connorl/Skeletons/For_Kevin/S32_Pos52,53,54_MIP0/Skeletons/Pos52_Skels.swc")
import_neurons(skels=skels, server="http://bigkahuna:4554/", api_token='7deede658f4bc1f53ee6993fc28ec26fbbe5838f', project_id = '', res=[.705,.812,.812])